In [2]:
import pandas as pd 
import duckdb as db

In [3]:
df=pd.read_csv("input/unlabeled_transactions.csv")
df.columns

Index(['Timestamp', 'From Bank', 'Account', 'To Bank', 'Account_duplicated_0',
       'Amount Received', 'Receiving Currency', 'Amount Paid',
       'Payment Currency', 'Payment Format'],
      dtype='object')

In [2]:
con = db.connect("data.duckdb")

con.execute("""
CREATE TABLE raw_data AS
SELECT *
FROM read_csv_auto('input/unlabeled_transactions.csv');
""")


In [3]:
con.execute("""
DESCRIBE SELECT *
FROM raw_data;
""").df()

,column_name,column_type,null,key,default,extra
0,Timestamp,TIMESTAMP,YES,None,None,None
1,From Bank,VARCHAR,YES,None,None,None
2,Account,VARCHAR,YES,None,None,None
3,To Bank,VARCHAR,YES,None,None,None
4,Account_duplicated_0,VARCHAR,YES,None,None,None
5,Amount Received,DOUBLE,YES,None,None,None
6,Receiving Currency,VARCHAR,YES,None,None,None
7,Amount Paid,DOUBLE,YES,None,None,None
8,Payment Currency,VARCHAR,YES,None,None,None
9,Payment Format,VARCHAR,YES,None,None,None


In [4]:
con.execute("""
select * from raw_data
""").df()

,Timestamp,From Bank,Account,To Bank,Account_duplicated_0,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format
0,2022-10-17 08:00:00,BankE,8000000X,BankC,8000005Y,9374.17,USD,9374.17,USD,Wire
1,2022-10-17 08:15:00,BankB,8000001X,BankC,8000006Y,1171.74,USD,1171.74,USD,Reinvestment
2,2022-10-17 08:30:00,BankE,8000002X,BankD,8000007Y,158.96,USD,158.96,USD,Cash
3,2022-10-17 08:45:00,BankE,8000003X,BankB,8000008Y,222.87,USD,222.87,USD,Cash
4,2022-10-17 09:00:00,BankD,8000004X,BankE,8000009Y,1085.78,USD,1085.78,USD,Wire
...,...,...,...,...,...,...,...,...,...,...
9995,2023-01-29 10:45:00,BankA,80009995X,BankB,8000000Y,258.70,USD,258.70,USD,ACH
9996,2023-01-29 11:00:00,BankE,80009996X,BankE,8000001Y,1063.67,USD,1063.67,USD,Reinvestment
9997,2023-01-29 11:15:00,BankD,80009997X,BankA,8000002Y,1192.29,USD,1192.29,USD,Reinvestment
9998,2023-01-29 11:30:00,BankB,80009998X,BankC,8000003Y,74.02,USD,74.02,USD,Reinvestment


In [10]:
def value():
    return con.execute("""
        SUMMARIZE raw_data
    """).df()

value()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,Timestamp,TIMESTAMP,2022-10-17 08:00:00,2023-01-29 11:45:00,8738,2022-12-08 09:52:30,None,2022-11-12 08:52:30,2022-12-08 09:52:30,2023-01-03 10:52:30,10000,0.0
1,From Bank,VARCHAR,BankA,BankE,5,None,None,None,None,None,10000,0.0
2,Account,VARCHAR,8000000X,8000999X,9703,None,None,None,None,None,10000,0.0
3,To Bank,VARCHAR,BankA,BankE,5,None,None,None,None,None,10000,0.0
4,Account_duplicated_0,VARCHAR,8000000Y,8000099Y,92,None,None,None,None,None,10000,0.0
5,Amount Received,DOUBLE,10.08,9998.64,10625,1625.348327000001,2662.35678147587,422.63974359400544,826.4031428605082,1236.6889294183236,10000,0.0
6,Receiving Currency,VARCHAR,USD,USD,1,None,None,None,None,None,10000,0.0
7,Amount Paid,DOUBLE,10.08,9998.64,10625,1625.348327000001,2662.35678147587,422.63974359400544,826.4031428605082,1236.6889294183236,10000,0.0
8,Payment Currency,VARCHAR,USD,USD,1,None,None,None,None,None,10000,0.0
9,Payment Format,VARCHAR,ACH,Wire,4,None,None,None,None,None,10000,0.0


In [15]:
def isnull_value():
    return con.execute("""
        SELECT
            SUM(CASE WHEN "Timestamp" IS NULL THEN 1 ELSE 0 END) as Timestamp,
            SUM(CASE WHEN "From Bank" IS NULL THEN 1 ELSE 0 END) as "From Bank",
            SUM(CASE WHEN "Account" IS NULL THEN 1 ELSE 0 END) as Account,
            SUM(CASE WHEN "To Bank" IS NULL THEN 1 ELSE 0 END) as "To Bank",
            SUM(CASE WHEN "Account_duplicated_0" IS NULL THEN 1 ELSE 0 END) as Account_duplicated_0,
            SUM(CASE WHEN "Amount Received" IS NULL THEN 1 ELSE 0 END) as "Amount Received",
            SUM(CASE WHEN "Receiving Currency" IS NULL THEN 1 ELSE 0 END) as "Receiving Currency",
            SUM(CASE WHEN "Amount Paid" IS NULL THEN 1 ELSE 0 END) as "Amount Paid",
            SUM(CASE WHEN "Payment Currency" IS NULL THEN 1 ELSE 0 END) as "Payment Currency",
            SUM(CASE WHEN "Payment Format" IS NULL THEN 1 ELSE 0 END) as "Payment Format"
        FROM raw_data
    """).df()

isnull_value()

,Timestamp,From Bank,Account,To Bank,Account_duplicated_0,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
